# Detection de Panneaux Routiers - Kaggle YOLO Notebook

This notebook clones the dataset from `https://github.com/OmarLabiade/ProjetAP`, shows dataset statistics and sample images, then trains an imported pretrained Ultralytics YOLO model.

Before running it on Kaggle:
- enable **Internet**
- enable **GPU** if available
- if Kaggle's CUDA stack is incompatible with the current runtime, the notebook will detect that and fall back to CPU instead of crashing


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "ultralytics"],
        check=True,
    )
    print("Installed ultralytics.")
else:
    print("ultralytics already available.")


## 1. Imports et parametres


In [ ]:
from __future__ import annotations

import math
import os
import random
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
import yaml
from PIL import Image

DATASET_REPO_URL = "https://github.com/OmarLabiade/ProjetAP.git"
DATASET_CLONE_DIR = Path("/kaggle/working/ProjetAP_repo")
DATASET_SUBDIR = "dataset"

CLASS_NAMES = [
    "complement",
    "danger",
    "direction",
    "indication",
    "localisation",
    "ordre",
]
COLORS = ["cyan", "red", "blue", "green", "orange", "magenta"]
NB_CLASSES = len(CLASS_NAMES)

MODEL_NAME = "yolov8l.pt"
IMAGE_SIZE = 1280  # Set to 0 for automatic choice based on training images.
MIN_AUTO_IMAGE_SIZE = 960
MAX_AUTO_IMAGE_SIZE = 1280

EPOCHS = 75
BATCH_SIZE = 4
WORKERS = 4
PREFERRED_DEVICE = "auto"  # "auto", "cpu", or "cuda:0"
ALLOW_CPU_FALLBACK = True
PATIENCE = 25

PROJECT_DIR = "/kaggle/working/runs_road_signs"
RUN_NAME = "yolo_git_kaggle"

OPTIMIZER = "AdamW"
LR0 = 1e-3
LRF = 1e-2
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 3.0

RECT = True
AMP = True
CLOSE_MOSAIC = 10
HSV_H = 0.010
HSV_S = 0.50
HSV_V = 0.35
TRANSLATE = 0.05
SCALE = 0.30
PERSPECTIVE = 0.0005
FLIPLR = 0.5
MOSAIC = 0.20

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


## 2. Recuperation et inspection du dataset


In [ ]:
def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def resolve_dataset_dir(root: Path) -> Path:
    if (
        (root / "train").is_dir()
        and ((root / "val").is_dir() or (root / "valid").is_dir())
        and (root / "test").is_dir()
    ):
        return root
    nested = root / DATASET_SUBDIR
    if (
        (nested / "train").is_dir()
        and ((nested / "val").is_dir() or (nested / "valid").is_dir())
        and (nested / "test").is_dir()
    ):
        return nested
    raise FileNotFoundError(f"Could not find train plus val/valid plus test under {root}")


def clone_dataset_repo(repo_url: str, clone_dir: Path) -> Path:
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    run(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
    return clone_dir


def get_split_dirs(dataset_root: Path) -> tuple[Path, Path, Path]:
    train_dir = dataset_root / "train"
    val_dir = dataset_root / "val" if (dataset_root / "val").is_dir() else dataset_root / "valid"
    test_dir = dataset_root / "test"
    return train_dir, val_dir, test_dir


def collect_images(folder: Path) -> list[Path]:
    images: list[Path] = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        images.extend(sorted(folder.glob(ext)))
    return images


def find_image_for_stem(stem_path: Path) -> Path | None:
    stem_str = str(stem_path)
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        candidate = Path(stem_str + ext)
        if candidate.exists():
            return candidate
    return None


def load_yolo_boxes(label_path: Path) -> list[list[float]]:
    boxes: list[list[float]] = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(float(parts[0]))
        cx, cy, w, h = map(float, parts[1:])
        boxes.append([cx, cy, w, h, cls_id])
    return boxes


def load_split_annotations(split_dir: Path) -> list[tuple[Path, list[list[float]]]]:
    pairs: list[tuple[Path, list[list[float]]]] = []
    for label_path in sorted(split_dir.glob("*.txt")):
        image_path = find_image_for_stem(label_path.with_suffix(""))
        if image_path is None:
            continue
        pairs.append((image_path, load_yolo_boxes(label_path)))
    return pairs


def round_to_stride(value: float, stride: int = 32) -> int:
    return int(math.ceil(value / stride) * stride)


def recommend_image_size(train_dir: Path) -> tuple[int, dict[str, float]]:
    image_paths = collect_images(train_dir)[:256]
    widths: list[int] = []
    heights: list[int] = []

    for image_path in image_paths:
        with Image.open(image_path) as image:
            widths.append(image.width)
            heights.append(image.height)

    if not widths:
        return 1280, {"median_width": 0.0, "median_height": 0.0, "median_aspect": 0.0}

    median_width = float(np.median(widths))
    median_height = float(np.median(heights))
    median_long_side = max(median_width, median_height)
    suggested = round_to_stride(
        min(MAX_AUTO_IMAGE_SIZE, max(MIN_AUTO_IMAGE_SIZE, median_long_side * 0.80))
    )
    return suggested, {
        "median_width": median_width,
        "median_height": median_height,
        "median_aspect": median_width / max(median_height, 1.0),
    }


def stats_dataset(pairs: list[tuple[Path, list[list[float]]]], name: str) -> None:
    counts = [0] * NB_CLASSES
    for _, boxes in pairs:
        for box in boxes:
            cls_id = int(box[4])
            if 0 <= cls_id < NB_CLASSES:
                counts[cls_id] += 1
    total = sum(counts)
    print(f"\n=== {name} - {len(pairs)} images, {total} objets ===")
    for idx, label in enumerate(CLASS_NAMES):
        ratio = 100.0 * counts[idx] / max(total, 1)
        print(f"  {label:15s}: {counts[idx]:4d} ({ratio:.1f}%)")
    plt.figure(figsize=(8, 3))
    plt.bar(CLASS_NAMES, counts, color=COLORS)
    plt.title(f"Distribution - {name}")
    plt.ylabel("Instances")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


def visualize_samples(pairs: list[tuple[Path, list[list[float]]]], title: str, n: int = 4) -> None:
    if not pairs:
        print(f"No samples available for {title}.")
        return

    sample_count = min(n, len(pairs))
    sample_pairs = random.sample(pairs, sample_count)
    fig, axes = plt.subplots(sample_count, 1, figsize=(14, 5 * sample_count))
    if sample_count == 1:
        axes = [axes]

    for ax, (image_path, boxes) in zip(axes, sample_pairs):
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        ax.imshow(image)
        for cx, cy, w, h, cls_id in boxes:
            x1 = (cx - w / 2.0) * width
            y1 = (cy - h / 2.0) * height
            rect = patches.Rectangle(
                (x1, y1),
                w * width,
                h * height,
                linewidth=2,
                edgecolor=COLORS[int(cls_id) % len(COLORS)],
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                x1,
                max(y1 - 4, 0),
                CLASS_NAMES[int(cls_id)],
                color=COLORS[int(cls_id) % len(COLORS)],
                fontsize=9,
                fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.45, pad=1),
            )
        ax.set_title(f"{title} - {image_path.name} - {len(boxes)} objets")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def build_runtime_yaml(dataset_root: Path, class_names: list[str], output_path: Path) -> Path:
    source_yaml = dataset_root / "data.yaml"
    runtime_config = {}
    _, val_dir, _ = get_split_dirs(dataset_root)

    if source_yaml.exists():
        with source_yaml.open("r", encoding="utf-8") as handle:
            runtime_config = yaml.safe_load(handle) or {}

    runtime_config["path"] = dataset_root.as_posix()
    runtime_config["train"] = "train"
    runtime_config["val"] = val_dir.name
    runtime_config["test"] = "test"
    runtime_config["nc"] = len(class_names)
    runtime_config["names"] = {idx: name for idx, name in enumerate(class_names)}

    with output_path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(runtime_config, handle, sort_keys=False, allow_unicode=True)

    return output_path


def resolve_training_device(preferred_device: str, allow_cpu_fallback: bool = True):
    if preferred_device == "cpu":
        print("Using CPU because PREFERRED_DEVICE='cpu'.")
        return "cpu"

    if not torch.cuda.is_available():
        print("CUDA is not available in this runtime. Using CPU.")
        return "cpu"

    target = "cuda:0" if preferred_device == "auto" else preferred_device

    try:
        device_index = int(str(target).split(":")[-1]) if ":" in str(target) else 0
        name = torch.cuda.get_device_name(device_index)
        capability = torch.cuda.get_device_capability(device_index)
        print(f"CUDA device detected: {name} (capability={capability})")

        with torch.no_grad():
            sample = torch.zeros((1, 3, 64, 64), device=target)
            probe = torch.nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1).to(target)
            _ = probe(sample)

        print(f"CUDA probe passed on {target}.")
        return device_index
    except Exception as exc:
        print(f"CUDA probe failed on {target}: {exc}")
        if not allow_cpu_fallback:
            raise
        print("Fallwithout touching Kaggle's torch stacking back to CPU.")
        return "cpu"


repo_root = clone_dataset_repo(DATASET_REPO_URL, DATASET_CLONE_DIR)
dataset_root = resolve_dataset_dir(repo_root)
train_dir, val_dir, test_dir = get_split_dirs(dataset_root)

train_pairs = load_split_annotations(train_dir)
val_pairs = load_split_annotations(val_dir)
test_pairs = load_split_annotations(test_dir)

train_images = collect_images(train_dir)
val_images = collect_images(val_dir)
test_images = collect_images(test_dir)

if len(train_images) < 2:
    raise RuntimeError("Need at least 2 training images for stable training.")

auto_image_size, dimension_summary = recommend_image_size(train_dir)
final_image_size = IMAGE_SIZE if IMAGE_SIZE else auto_image_size
runtime_device = resolve_training_device(PREFERRED_DEVICE, ALLOW_CPU_FALLBACK)
yaml_path = build_runtime_yaml(dataset_root, CLASS_NAMES, Path("/kaggle/working/dataset_ultralytics.yaml"))

print(f"Repo root          : {repo_root}")
print(f"Dataset root       : {dataset_root}")
print(f"Train split        : {train_dir}")
print(f"Validation split   : {val_dir}")
print(f"Test split         : {test_dir}")
print(f"Train images       : {len(train_images)}")
print(f"Validation images  : {len(val_images)}")
print(f"Test images        : {len(test_images)}")
print(f"Median train size  : {dimension_summary['median_width']:.0f}x{dimension_summary['median_height']:.0f}")
print(f"Median aspect ratio: {dimension_summary['median_aspect']:.3f}")
print(f"Auto image size    : {auto_image_size}")
print(f"Chosen image size  : {final_image_size}")
print(f"Runtime device     : {runtime_device}")
print(f"Data yaml          : {yaml_path}")

stats_dataset(train_pairs, "Train")
stats_dataset(val_pairs, "Validation")
stats_dataset(test_pairs, "Test")


## 3. Visualisation des donnees


In [ ]:
visualize_samples(train_pairs, "Train", n=4)
visualize_samples(val_pairs, "Validation", n=2)


## 4. Entrainement du modele importe


In [ ]:
from ultralytics import YOLO
from ultralytics.data.build import build_dataloader
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.torch_utils import torch_distributed_zero_first
import torch

DEVICE = "0"

print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Training will fail because DEVICE='0' forces GPU.")


class DropLastTrainBatchTrainer(DetectionTrainer):
    def get_dataloader(self, dataset_path: str, batch_size: int = 16, rank: int = 0, mode: str = "train"):
        assert mode in {"train", "val"}, f"Mode must be 'train' or 'val', not {mode}."
        with torch_distributed_zero_first(rank):
            dataset = self.build_dataset(dataset_path, mode, batch_size)
        shuffle = mode == "train"
        if getattr(dataset, "rect", False) and shuffle and not np.all(dataset.batch_shapes == dataset.batch_shapes[0]):
            shuffle = False
        return build_dataloader(
            dataset,
            batch=batch_size,
            workers=self.args.workers if mode == "train" else self.args.workers * 2,
            shuffle=shuffle,
            rank=rank,
            drop_last=(mode == "train"),
        )


model = YOLO(MODEL_NAME)

train_kwargs = dict(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=final_image_size,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    pretrained=True,
    cache=False,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=3.0,
    cos_lr=True,
    patience=PATIENCE,
    close_mosaic=10,
    amp=True,
    rect=True,
    multi_scale=False,
    hsv_h=0.010,
    hsv_s=0.50,
    hsv_v=0.35,
    degrees=0.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.20,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    overlap_mask=False,
    plots=True,
    save=True,
    val=True,
    verbose=True,
)

results = model.train(trainer=DropLastTrainBatchTrainer, **train_kwargs)

print("\nTraining finished.")
print(f"Run directory: {results.save_dir}")

best_model_path = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_model_path if best_model_path.exists() else MODEL_NAME))

print("\nValidating best checkpoint on validation split...")
best_model.val(data=str(yaml_path), split="val", imgsz=final_image_size, device=DEVICE, rect=True, plots=True)

print("\nValidating best checkpoint on test split...")
best_model.val(data=str(yaml_path), split="test", imgsz=final_image_size, device=DEVICE, rect=True, plots=True)

## 5. Evaluation


In [ ]:
best_model_path = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_model_path if best_model_path.exists() else MODEL_NAME))

print("Validating best checkpoint on validation split...")
val_metrics = best_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=final_image_size,
    device=runtime_device,
    rect=True,
    plots=True,
)

print("Validating best checkpoint on test split...")
test_metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=final_image_size,
    device=runtime_device,
    rect=True,
    plots=True,
)

print("Best weights:", best_model_path)
print("Validation metrics:", val_metrics.results_dict)
print("Test metrics:", test_metrics.results_dict)


## Prediction

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def show_prediction_on_random_example(model, split_dir, split_name="Test", conf=0.25):
    image_paths = collect_images(split_dir)
    if not image_paths:
        print(f"No images found in {split_dir}")
        return

    image_path = random.choice(image_paths)
    results = model.predict(
        source=str(image_path),
        imgsz=final_image_size,
        conf=conf,
        device=DEVICE,
        verbose=False,
    )

    result = results[0]
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(image)

    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        scores = result.boxes.conf.cpu().numpy()

        for box, cls_id, score in zip(boxes_xyxy, classes, scores):
            x1, y1, x2, y2 = box
            color = COLORS[cls_id % len(COLORS)]
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                x1,
                max(y1 - 5, 0),
                f"{CLASS_NAMES[cls_id]} {score:.2f}",
                color=color,
                fontsize=10,
                fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.45, pad=1),
            )
    else:
        ax.text(
            20,
            30,
            "No detections",
            color="white",
            fontsize=12,
            fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.6, pad=4),
        )

    ax.set_title(f"{split_name} prediction - {image_path.name}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


best_model_path = Path(results.save_dir) / "weights" / "best.pt"
prediction_model = YOLO(str(best_model_path if best_model_path.exists() else MODEL_NAME))

show_prediction_on_random_example(prediction_model, test_dir, split_name="Test", conf=0.25)

**Evaluation**


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import random
from PIL import Image

CONFIDENCE_THRESHOLD = 0.20
IOU_THRESHOLD = 0.50


def iou(b1, b2):
    x1 = max(b1[0] - b1[2] / 2, b2[0] - b2[2] / 2)
    y1 = max(b1[1] - b1[3] / 2, b2[1] - b2[3] / 2)
    x2 = min(b1[0] + b1[2] / 2, b2[0] + b2[2] / 2)
    y2 = min(b1[1] + b1[3] / 2, b2[1] + b2[3] / 2)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = b1[2] * b1[3] + b2[2] * b2[3] - inter
    return inter / union if union > 0 else 0.0


def apply_classwise_nms(boxes, iou_threshold=0.50):
    kept = []
    for cls_id in range(NB_CLASSES):
        cls_boxes = [b for b in boxes if int(b[4]) == cls_id]
        cls_boxes = sorted(cls_boxes, key=lambda b: b[5], reverse=True)
        while cls_boxes:
            best = cls_boxes.pop(0)
            kept.append(best)
            cls_boxes = [b for b in cls_boxes if iou(best[:4], b[:4]) < iou_threshold]
    return kept


def predict_split(model, split_dir, conf=0.20):
    image_paths = collect_images(split_dir)
    y_pred_list = []

    for image_path in image_paths:
        results = model.predict(
            source=str(image_path),
            imgsz=final_image_size,
            conf=conf,
            device=DEVICE,
            verbose=False,
        )

        result = results[0]
        image = Image.open(image_path)
        width, height = image.size

        pred_boxes = []
        if result.boxes is not None and len(result.boxes) > 0:
            boxes_xyxy = result.boxes.xyxy.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            scores = result.boxes.conf.cpu().numpy()

            for box, cls_id, score in zip(boxes_xyxy, classes, scores):
                x1, y1, x2, y2 = box
                cx = ((x1 + x2) / 2.0) / width
                cy = ((y1 + y2) / 2.0) / height
                w = (x2 - x1) / width
                h = (y2 - y1) / height
                pred_boxes.append([cx, cy, w, h, cls_id, float(score)])

        y_pred_list.append(apply_classwise_nms(pred_boxes, iou_threshold=0.50))

    return image_paths, y_pred_list


def extract_gt_boxes(pairs, image_paths):
    gt_by_name = {img_path.name: boxes for img_path, boxes in pairs}
    return [gt_by_name[path.name] for path in image_paths]


def compute_metrics(y_true_list, y_pred_list, iou_thres=0.5):
    TP = [0] * NB_CLASSES
    FP = [0] * NB_CLASSES
    FN = [0] * NB_CLASSES
    iou_sum = [0.0] * NB_CLASSES
    iou_cnt = [0] * NB_CLASSES

    for gt_b, pd_b in zip(y_true_list, y_pred_list):
        matched = set()

        for p in pd_b:
            pc = int(p[4])
            best, bj = 0.0, -1

            for j, g in enumerate(gt_b):
                if j in matched or int(g[4]) != pc:
                    continue
                v = iou(g[:4], p[:4])
                if v > best:
                    best, bj = v, j

            if best >= iou_thres and bj >= 0:
                TP[pc] += 1
                matched.add(bj)
                iou_sum[pc] += best
                iou_cnt[pc] += 1
            else:
                FP[pc] += 1

        for j, g in enumerate(gt_b):
            if j not in matched:
                FN[int(g[4])] += 1

    res = []
    for i in range(NB_CLASSES):
        P = TP[i] / (TP[i] + FP[i]) if TP[i] + FP[i] > 0 else 0.0
        R = TP[i] / (TP[i] + FN[i]) if TP[i] + FN[i] > 0 else 0.0
        F1 = 2 * P * R / (P + R) if P + R > 0 else 0.0
        mI = iou_sum[i] / iou_cnt[i] if iou_cnt[i] > 0 else 0.0
        res.append({"Precision": P, "Recall": R, "F1": F1, "mIoU": mI})

    acc = 100.0 * sum(TP) / max(sum(TP) + sum(FP), 1)
    miou = np.mean([r["mIoU"] for r in res if r["mIoU"] > 0]) if any(r["mIoU"] > 0 for r in res) else 0.0
    return res, acc, miou


def afficher_metriques(res, acc, miou):
    print(f"\nPrecision globale : {acc:.1f}%  -  mIoU : {miou:.3f}")
    print(f"{'Classe':15s} | {'Precision':9s} | {'Recall':7s} | {'F1':7s} | {'mIoU':7s}")
    print("-" * 58)
    for i, r in enumerate(res):
        print(f"{CLASS_NAMES[i]:15s} | {r['Precision']:9.3f} | {r['Recall']:7.3f} | {r['F1']:7.3f} | {r['mIoU']:7.3f}")


def visualiser_predictions(image_paths, y_true, y_pred, indices=None, nb=4):
    if indices is None:
        indices = np.random.choice(len(image_paths), min(nb, len(image_paths)), replace=False)

    for idx in indices:
        image = np.array(Image.open(image_paths[idx]).convert("RGB"))
        height, width = image.shape[:2]

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, boxes, title in zip(axes, [y_true[idx], y_pred[idx]], ["Verite terrain", "Prediction"]):
            ax.imshow(image)
            for b in boxes:
                cx, cy, w, h, cid = b[0], b[1], b[2], b[3], int(b[4])
                conf = f" {b[5]:.2f}" if len(b) > 5 else ""
                rect = patches.Rectangle(
                    ((cx - w / 2) * width, (cy - h / 2) * height),
                    w * width,
                    h * height,
                    linewidth=2,
                    edgecolor=COLORS[cid],
                    facecolor="none",
                )
                ax.add_patch(rect)
                ax.text(
                    (cx - w / 2) * width,
                    max((cy - h / 2) * height - 3, 0),
                    CLASS_NAMES[cid] + conf,
                    color=COLORS[cid],
                    fontsize=8,
                    fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.4, pad=1),
                )
            ax.set_title(title)
            ax.axis("off")
        plt.tight_layout()
        plt.show()


def build_confusion_matrix(y_true_list, y_pred_list, iou_thres=0.5):
    confusion = np.zeros((NB_CLASSES, NB_CLASSES), dtype=int)

    for gt_b, pd_b in zip(y_true_list, y_pred_list):
        for p in pd_b:
            pc = int(p[4])
            best, bgc = 0.0, -1
            for g in gt_b:
                v = iou(g[:4], p[:4])
                if v > best:
                    best, bgc = v, int(g[4])
            if best >= iou_thres and bgc >= 0:
                confusion[bgc, pc] += 1

    return confusion

In [ ]:
best_model_path = Path(results.save_dir) / "weights" / "best.pt"
prediction_model = YOLO(str(best_model_path if best_model_path.exists() else MODEL_NAME))

test_image_paths, y_pred_test = predict_split(
    prediction_model,
    test_dir,
    conf=CONFIDENCE_THRESHOLD,
)

y_true_test = extract_gt_boxes(test_pairs, test_image_paths)

res, acc, miou = compute_metrics(y_true_test, y_pred_test, iou_thres=IOU_THRESHOLD)
afficher_metriques(res, acc, miou)

In [ ]:
print("=== Exemples aleatoires ===")
visualiser_predictions(test_image_paths, y_true_test, y_pred_test, nb=6)

bons = [
    i for i, (g, p) in enumerate(zip(y_true_test, y_pred_test))
    if any(int(gt[4]) == int(pd[4]) and iou(gt[:4], pd[:4]) >= 0.5 for gt in g for pd in p)
]

mauvais = [
    i for i, (g, p) in enumerate(zip(y_true_test, y_pred_test))
    if g and any(
        not any(int(pd[4]) == int(gt[4]) and iou(gt[:4], pd[:4]) >= 0.5 for pd in p)
        for gt in g
    )
]

print(f"Bons exemples : {len(bons)}  -  Mauvais exemples : {len(mauvais)}")

print("\n=== Bons exemples ===")
visualiser_predictions(test_image_paths, y_true_test, y_pred_test, indices=bons[:4])

print("\n=== Mauvais exemples ===")
visualiser_predictions(test_image_paths, y_true_test, y_pred_test, indices=mauvais[:4])

In [ ]:
confusion = build_confusion_matrix(y_true_test, y_pred_test, iou_thres=IOU_THRESHOLD)

plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Classe predite")
plt.ylabel("Classe reelle")
plt.title("Matrice de confusion - test (IoU >= 0.5)")
plt.tight_layout()
plt.show()

In [ ]:
for name, split_dir, pairs in [
    ("Train", train_dir, train_pairs),
    ("Validation", val_dir, val_pairs),
    ("Test", test_dir, test_pairs),
]:
    image_paths, y_pred = predict_split(prediction_model, split_dir, conf=CONFIDENCE_THRESHOLD)
    y_true = extract_gt_boxes(pairs, image_paths)
    r, a, m = compute_metrics(y_true, y_pred, iou_thres=IOU_THRESHOLD)

    print(f"\n{'=' * 50}  {name}")
    afficher_metriques(r, a, m)